<a href="https://colab.research.google.com/github/huutai-cmyk/Homework-1/blob/main/apptiendien.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import gradio as gr
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import re

# ==========================================
# 1. ĐỌC DỮ LIỆU & HUẤN LUYỆN AI (Giữ nguyên logic cũ)
# ==========================================
# Đọc file CSV bạn đã tải lên Colab
df_raw = pd.read_csv('/Tiền điện - Câu trả lời biểu mẫu 1.csv')

df = df_raw.rename(columns={
    'Số người trong phòng là bao nhiêu?': 'SoNguoi',
    'Số máy lạnh ở phòng là bao nhiêu?': 'SoMayLanh',
    'Phòng có tủ lạnh không?': 'CoTuLanh',
    'Số giờ bật máy lạnh trên ngày?': 'GioMayLanh',
    'Diện tích phòng khoảng bao nhiêu m2?': 'DienTich',
    'Tiền điện trung bình 1 tháng là bao nhiêu vnd?': 'TienDien'
})

df['SoQuat'] = df['Số quạt máy và thời gian sử dụng 1 ngày?'].str.extract('(\d+)').astype(int)

def clean_loainha(x):
    x = str(x).strip().lower()
    if 'trọ' in x: return 'Trọ/Phòng trọ'
    if 'chung cư' in x: return 'Chung cư'
    if 'căn hộ' in x: return 'Căn hộ'
    if 'nhà nguyên căn' in x: return 'Nhà nguyên căn'
    if 'túc xá' in x: return 'Ký túc xá'
    return 'Khác'

df['LoaiNha'] = df['Loại nhà? (trọ,chung cư, căn hộ…)'].apply(clean_loainha)
df = df[['SoNguoi', 'SoMayLanh', 'SoQuat', 'CoTuLanh', 'GioMayLanh', 'DienTich', 'LoaiNha', 'TienDien']]

X = df.drop('TienDien', axis=1)
y = df['TienDien']

numeric_features = ['SoNguoi', 'SoMayLanh', 'SoQuat', 'GioMayLanh', 'DienTich']
categorical_features = ['CoTuLanh', 'LoaiNha']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Rừng ngẫu nhiên (Random Forest)
model = Pipeline(steps=[('preprocessor', preprocessor),
                        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])
model.fit(X, y)

# ==========================================
# 2. XÂY DỰNG GIAO DIỆN APP ĐIỆN THOẠI VỚI GRADIO
# ==========================================

# Hàm này sẽ được Gradio gọi mỗi khi người dùng bấm nút "Dự đoán"
def predict_bill(songuoi, somaylanh, soquat, cotulanh, giomaylanh, dientich, loainha):
    # Nếu không có máy lạnh thì số giờ = 0
    giomaylanh_thucte = giomaylanh if somaylanh > 0 else 0

    input_data = pd.DataFrame({
        'SoNguoi': [songuoi],
        'SoMayLanh': [somaylanh],
        'SoQuat': [soquat],
        'CoTuLanh': [cotulanh],
        'GioMayLanh': [giomaylanh_thucte],
        'DienTich': [dientich],
        'LoaiNha': [loainha]
    })

    # AI dự đoán
    prediction = model.predict(input_data)[0]

    # Trả về chuỗi kết quả có định dạng VNĐ
    return f"{prediction:,.0f} VNĐ"

# Cấu hình các ô nhập liệu giống hệt app mobile
inputs = [
    gr.Slider(minimum=1, maximum=10, step=1, value=2, label="👥 Số người ở"),
    gr.Slider(minimum=0, maximum=5, step=1, value=1, label="❄️ Số máy lạnh"),
    gr.Slider(minimum=0, maximum=10, step=1, value=2, label="🎐 Số quạt"),
    gr.Radio(choices=["Có", "Không"], value="Có", label="🧊 Có tủ lạnh không?"),
    gr.Slider(minimum=0, maximum=24, step=1, value=4, label="⏱️ Số giờ bật máy lạnh/ngày"),
    gr.Number(value=25, label="📐 Diện tích phòng (m2)"),
    gr.Dropdown(
        choices=['Trọ/Phòng trọ', 'Chung cư', 'Căn hộ', 'Nhà nguyên căn', 'Ký túc xá'],
        value='Trọ/Phòng trọ',
        label="🏠 Loại nhà"
    )
]

# Cấu hình ô hiển thị kết quả
output = gr.Textbox(label="💡 ƯỚC TÍNH TIỀN ĐIỆN MỖI THÁNG", lines=2)

# Lắp ráp App
app = gr.Interface(
    fn=predict_bill,
    inputs=inputs,
    outputs=output,
    title="⚡ APP DỰ ĐOÁN TIỀN ĐIỆN",
    description="Nhập thông tin thiết bị phòng trọ của bạn để AI dự đoán số tiền điện tháng tới nhé!",
    theme=gr.themes.Soft() # Dùng giao diện bo góc mềm mại giống iOS/Android
)

# Khởi chạy App và tạo link Public để mở trên điện thoại
app.launch(share=True)

<>:25: SyntaxWarning: invalid escape sequence '\d'
<>:25: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_15494/1072037884.py:25: SyntaxWarning: invalid escape sequence '\d'
  df['SoQuat'] = df['Số quạt máy và thời gian sử dụng 1 ngày?'].str.extract('(\d+)').astype(int)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0aa63de96def1ea7b3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Mục mới